In [1]:
!pip install pypdf


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

load_dotenv()

c:\Users\bunga\projek_data_science\agentic_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
# Singleton holder for the embeddings instance
_embeddings_instance: HuggingFaceEmbeddings | None = None

def get_hf_embeddings(
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
) -> HuggingFaceEmbeddings:
    """
    Return a cached HuggingFaceEmbeddings instance for `model_name`.
    On first call it will download (if needed), later calls reuse the same object.
    """
    global _embeddings_instance
    if _embeddings_instance is None:
        # This will download the model into your HF cache dir if not already present
        _embeddings_instance = HuggingFaceEmbeddings(model_name=model_name)
    return _embeddings_instance


model = get_hf_embeddings()

In [6]:
# extract data  from pdf file
def load_pdf(data):
    loader = DirectoryLoader(data, # <- directory file
                             glob="*.pdf", # <- all pdf on folder
                             loader_cls=PyPDFLoader # <- function implement to load pdf
                             )
    documents = loader.load()
    return documents


#function chucking documents
def text_splitter(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks   = text_splitter.split_documents(extracted_data)
    return text_chunks

In [7]:
document = load_pdf('../data/')

In [8]:
type(document)

list

In [9]:
text_chunk = text_splitter(document)
len(text_chunk)

5859

In [10]:
type(text_chunk)

list

In [11]:
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
PINECONE_API_KEY

'pcsk_4tHjw7_7pfG9K5vNeKEZaYtRcHrkjAeRPVNYBg3JjcePkC8vNZzAuA2ZRSFFqJoysuRQii'

In [12]:
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = 'medicalbot'


# pc.create_index(
#     name=index_name,
#     dimension=384,
#     metric='cosine',
#     spec=ServerlessSpec(
#         cloud='aws',
#         region="us-east-1"
#     )
# )

In [13]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-index"

# create index (kalau belum ada)
if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,  # sesuaikan embedding
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


In [20]:
# # pc = Pinecone(api_key=PINECONE_API_KEY)

# import pinecone
# from langchain_community.vectorstores import Pinecone

# # pinecone.init(
# #     api_key=PINECONE_API_KEY,
# #     environment="us-east-1"
# # )

# docsearch = Pinecone.from_documents(
#     documents=text_chunk,
#     index_name=index_name,
#     embedding=model
# )


from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)


docsearch = PineconeVectorStore.from_documents(
documents=text_chunk,
embedding=model,
index_name="medical-index"
)


In [21]:
#load existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=model
)

In [22]:
#setting retiever
rertriever = docsearch.as_retriever(search_type='similarity', search_kwargs={'k':3})

In [23]:
#testing
rertriever.invoke("What is acne ?")

[Document(id='048714b9-364a-4045-a523-501b82c07e44', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '..\\data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='2b76ceeb-7d02-4ba6-bda2-170de840f39f', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '..\\data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission

In [18]:
from langchain_groq import ChatGroq
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')


#define groq llm
llm = ChatGroq(temperature=0.5,
                groq_api_key=GROQ_API_KEY, 
                model_name="llama-3.3-70b-versatile")

# from langchain_google_genai import ChatGoogleGenerativeAI
# import os
# load_dotenv()


# llm = ChatGoogleGenerativeAI(
#     model="gemini-3-flash-preview",
#     temperature=0.5,
#     google_api_key=os.environ["GOOGLE_API_KEY"],
# )
llm.invoke("Hello, world!")

AIMessage(content="Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 39, 'total_tokens': 64, 'completion_time': 0.060436297, 'completion_tokens_details': None, 'prompt_time': 0.006234982, 'prompt_tokens_details': None, 'queue_time': 0.23790773, 'total_time': 0.066671279}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec006-ec04-7d13-a31d-c4412d87931c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 25, 'total_tokens': 64})

In [19]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

ModuleNotFoundError: No module named 'langchain.chains'

In [8]:
system_prompt = (
    "You are an asistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you dot't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [16]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(rertriever, question_answer_chain)

In [17]:
response = rag_chain.invoke({
        "input":"What is definition for Doppler?"
    })

print(response['answer'])

The Doppler effect refers to the apparent change in frequency of sound wave echoes returning to a stationary source from a moving target. This change in frequency can be used to compute the object's speed, whether it's a car or blood in an artery. The Doppler effect holds true for all types of radiation, not just sound.


In [18]:
response = rag_chain.invoke({
        "input":"What is machine learning?"
    })

print(response['answer'])

I don't know what machine learning is based on the provided context, as it doesn't mention machine learning. The context appears to be related to human memory, neurodegenerative diseases, and cancer treatments.
